#### Importación de librerías necesarias

In [ ]:
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

import os
import sys
sys.path.insert(0, os.path.abspath("../"))
import src.visualizaciones as visualizaciones

from IPython.display import display

# omite advertencias
import warnings
warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import hashlib

#### Carga de Data

In [ ]:
df = pd.read_csv("../data/Telco_Customer_Churn_Dataset.csv")

#### Información inicial de Data

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

* Este conjunto de datos consiste en 7043 registros y 21 características
* El dataset tiene muchas características como data de texto y probablemente sean variables categóricas
* **TotalCharges** contiene valores numéricos pero está almacenada como strings. Vamos a convertir esta variable a tipo numérica (float)
* La variable **SeniorCitizen** indica si el cliente es adulto mayor (0 = "no", 1 = "sí"). Al ser una variable categórica, la convertiremos de tipo numérico a object

# Manipulación de Datos

In [ ]:
# convertir columnas de tipo string a object
for columna in df.select_dtypes(include=['string']).columns:
        df[columna] = df[columna].astype('object')
print(f"Total de columnas convertidas a object: {len(df.select_dtypes(include=['object']).columns)}")

In [ ]:
# convertir TotalCharges a numérico
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("TotalCharges convertido a numérico.")

In [ ]:
# convertir SeniorCitizen a object para que sea categórico
df['SeniorCitizen'] = df['SeniorCitizen'].replace({0: '0', 1: '1'}).astype(object)
print("SeniorCitizen convertido a object.")

In [ ]:
# información general del DataFrame
df.info()

* La variable objetivo que guiará nuestra exploración es el **Churn**.

#### Implementación de Anonimización

* De acuerdo con la legislación sobre protección de la vida privada, almacenar de forma directa el ID del cliente (CustomerID) junto con su comportamiento financiero expone a la empresa a severas multas.

In [ ]:
def anonimizar(texto):
    # Genera un Hash único para el nombre (Irreversible)
    return hashlib.sha256(texto.encode()).hexdigest()

# Aplicar a la columna sensible
df['customerID_Hash'] = df['customerID'].apply(anonimizar)

# Eliminar el dato original
df.drop('customerID', axis=1, inplace=True)

In [ ]:
df.head()

# Análisis Exploratorio de Datos

#### Análisis de completitud

In [ ]:
# conteo de valores nulos por columna
df.isnull().sum()

* La columna **TotalCharges** tienen algunos valores faltantes

In [ ]:
# muestra registros con valores nulos en TotalCharges
df[np.isnan(df['TotalCharges'])]

* En los registros donde **TotalCharges** presenta valores nulos, la antiguedad del cliente (**tenure**) es igual a 0. Por este motivo, y dado que la baja cantidad de valores faltantes no afectará la calidad del conjunto, es seguro reemplazar dichos valores faltantes por 0

In [ ]:
# reemplazar valores nulos de TotalCharges por 0
df["TotalCharges"] = df["TotalCharges"].fillna(0)
print("Valores nulos de TotalCharges reemplazados por 0.")

#### Análisis de valores duplicados

In [ ]:
# registros duplicados
print(f"Cantidad de registros duplicados: {df.duplicated().sum()}")

* Tras la revisión, no se encontraron valores duplicados en el conjunto de datos

#### Análisis de valores atípicos

In [ ]:
# buscar valores atípicos usando el rango intercuartil
columnas_numericas = ["tenure", "MonthlyCharges", "TotalCharges"]

for columna in columnas_numericas:
    q1 = df[columna].quantile(0.25)
    q3 = df[columna].quantile(0.75)
    rango_intercuartil = q3 - q1

    limite_inferior = q1 - 1.5 * rango_intercuartil
    limite_superior = q3 + 1.5 * rango_intercuartil

    valores_atipicos = df[
        (df[columna] < limite_inferior) | (df[columna] > limite_superior)
    ]

    if len(valores_atipicos) > 0:
        print(f"{columna}: sí presenta valores atípicos ({len(valores_atipicos)} registros).")
    else:
        print(f"{columna}: no presenta valores atípicos.")

In [ ]:
# crear una figura con múltiples espacios
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Análisis visual de valores atípicos", fontsize=16, fontweight="bold")

# dibujar cada gráfico de caja con nombres descriptivos en español
sns.boxplot(y=df["tenure"], ax=axes[0])
axes[0].set_title("Antiguedad del cliente")
axes[0].set_xlabel("")
axes[0].set_ylabel("Antiguedad (meses)")

sns.boxplot(y=df["MonthlyCharges"], ax=axes[1])
axes[1].set_title("Cargos mensuales")
axes[1].set_xlabel("")
axes[1].set_ylabel("Cargos mensuales ($)")

sns.boxplot(y=df["TotalCharges"], ax=axes[2])
axes[2].set_title("Cargos totales")
axes[2].set_xlabel("")
axes[2].set_ylabel("Cargos totales ($)")

plt.subplots_adjust(wspace=0.35, top=0.82)
plt.show()

* No se encontraron valores atípicos en las variables numéricas analizadas

#### Estadísticas básicas

In [ ]:
# estadísticas descriptivas de las variables categóricas
df.describe(include="object").T.drop("customerID", axis=0)

Hallazgos relevantes:

* La distribución por género es equilibrada: hay 3555 hombres y 3488 mujeres
* La clase positiva **Churn = Yes**, que representa a los clientes que sí abandonan el servicio, contiene 1869 registros (26,5%)
* La mayoría de los clientes no son adultos mayores: **SeniorCitizen** = 0 representa aproximadamente el 83,8%
* El contrato más frecuente es mensual, con 3875 clientes, lo que representa aproximadamente el 55%
* Cerca del 90,3% de los clientes tiene servicio telefónico

In [ ]:
# estadísticas descriptivas de variables numéricas
df.describe(include=['int64', 'float64']).T

- La antiguedad mediana es de 29 meses, mientras que la media es de 32,37 meses
- Los cargos mensuales tienen una mediana de $70,35, con valores entre $18,25 y $118,75
- **TotalCharges** presenta una mediana de $1394,55 y una media de $2279,73

## Análisis Descriptivo

#### Variable Objetivo

In [ ]:
fig, ax = visualizaciones.graficar_distribucion_churn(
    df,
    ruta_salida="../resultados/plots/distribucion_variable_objetivo.png"
)
plt.show()

* La tasa general de churn es del 26.5%. Veamos entonces los distintos grupos de clientes. ¿Qué grupos presentan una mayor probabilidad de abandono?

### Características Numéricas

Para las variables numéricas, se calculan y grafican las funciones de estimación de densidad de kernel (KDE).

In [ ]:
# generar curvas de densidad según la variable Churn
fig, axes = visualizaciones.graficar_curvas_densidad_churn(
    df,
    ruta_salida="../resultados/plots/curvas_densidad_churn.png"
)
plt.show()

In [ ]:
# mapa de calor matriz de correlaciones variables numéricas
df_copy = df.copy()
df_copy["Churn"] = df_copy["Churn"].map({"No": 0, "Yes": 1}).astype(object)
df_copy["Churn"] = pd.to_numeric(df_copy["Churn"], errors="coerce")
numeric_df = df_copy.select_dtypes(include=[np.number])
numeric_df = numeric_df.drop(columns=['customerID'], errors='ignore')
correlation_matrix = numeric_df.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlación - Telco Customer Churn')
plt.show()

* Antiguedad y Cargos Totales (0,825): Fuerte correlación positiva. Como es lógico, los clientes que llevan más tiempo en la empresa acumulan un gasto total mayor.

* Antiguedad y Cargos Mensuales (0,248): Correlación positiva débil. Existe una ligera tendencia donde los clientes más antiguos asumen tarifas mensuales un poco más altas, lo que sugiere que podrían estar contratando servicios de mayor precio.

In [ ]:
# agrupar por 'Churn' y calcular el promedio
resumen = df.groupby('Churn')[['tenure', 'MonthlyCharges', 'TotalCharges']].mean().round(2)
tabla_churn_numeric = resumen.T
tabla_churn_numeric.columns = ['Churn = No', 'Churn = Sí']
tabla_churn_numeric

Se pueden extraer las siguientes conclusiones:

**Antiguedad (Meses):**

* Los clientes activos tienen una antiguedad promedio de 37,57 meses.
* Quienes cancelaron el servicio muestran una permanencia menor, con un promedio de 17,98 meses.
* A mayor tiempo en la compañía, menor es la probabilidad de abandono.
* Los clientes nuevos con poca antiguedad tienen más probabilidades de cambiar de compañía.

**Cargos Mensuales:**

* El pago mensual promedio de los clientes activos es de $61,27.
* Quienes se dieron de baja pagaban una tarifa un poco mayor, cercana a los $74,44.
* Los clientes que pagan cargos mensuales más altos también tienen más probabilidades de cancelar el servicio.

**Cargos Totales:**

* Los clientes que mantienen su suscripción acumulan un promedio de $2554,77 en cargos totales.
* Quienes cancelaron el servicio acumularon un gasto inferior, de alrededor de $1531,80.
* Los usuarios con un mayor gasto total acumulado en el tiempo son más propensos a quedarse en la empresa.

### Variables Categóricas

Para las variables categóricas, los gráficos de barras muestran las diferencias entre los grupos objetivo.

#### Información de Clientes

In [ ]:
figura_cliente = visualizaciones.graficar_grupo_categorico_churn(
    df,
    nombre_grupo="Información del cliente",
    ruta_salida="../resultados/plots"
)
display(figura_cliente)
plt.close(figura_cliente)

* La cantidad de cancelaciones es muy similar entre clientes hombres y mujeres, sin presentar una diferencia importante.
* Clientes sin pareja y clientes sin familiares (dependientes), así como los adultos mayores en proporción tiene mayor probabilidad de abono.
* La mayor parte de los clientes que son adultos mayores termina cancelando su servicio.

#### Servicios Contratados

In [ ]:
figura_servicios = visualizaciones.graficar_grupo_categorico_churn(
    df,
    nombre_grupo="Servicios contratados",
    ruta_salida="../resultados/plots"
)
display(figura_servicios)
plt.close(figura_servicios)

* Un alto número de clientes con internet de fibra óptica cancelaron su servicio.
* Los usuarios con conexión DSL abandonan la compañía con menor frecuencia.
* Los clientes que no cuentan con servicio de internet presentan una tasa de abandono muy baja.
* Una gran parte de los clientes elige el servicio de fibra óptica, pero presentan una alta tasa de abandono, lo que sugiere una posible insatisfacción con este tipo de conexión.

#### Seguridad y soporte

In [ ]:
figura_seguridad = visualizaciones.graficar_grupo_categorico_churn(
    df,
    nombre_grupo="Seguridad y soporte",
    ruta_salida="../resultados/plots"
)
display(figura_seguridad)
plt.close(figura_seguridad)

* Seguridad en línea y Soporte técnico son los servicios que más protegen la retención, bajando el abandono de un 42% (clientes sin el servicio) a solo un 15% (clientes con el servicio).
* Respaldo y Protección del dispositivo también muestran un impacto positivo logrando reducir el nivel de fuga de un 40% a un 22%.
* Los clientes que contrataron servicios de seguridad o soporte técnico adicional tienen menos probabilidades de cancelar su servicio. Resulta clave ofrecer promociones que incluyan seguridad o soporte técnico.

#### Información de Pago

In [ ]:
figura_pago = visualizaciones.graficar_grupo_categorico_churn(
    df,
    nombre_grupo="Información de pago",
    ruta_salida="../resultados/plots"
)
display(figura_pago)
plt.close(figura_pago)

* El abandono es crítico en la modalidad "Mes a mes" (42.7%), pero disminuye drásticamente en compromisos de "Un año" (11.2%) y "Dos años" (2.8%).
* Los clientes con facturación electrónica presentan el doble de tasa de abandono (33.5%) que aquellos con facturación tradicional (16.3%).
* El "Cheque electrónico" es el método de mayor riesgo con un 45.2% de fuga, mientras que las otras alternativas promedian un abandono mucho menor (15-19%).


* Es prioritario incentivar contratos largos. Ofrecer beneficios para que los clientes mensuales pasen a planes anuales o de dos años, donde la tasa de abandono es casi nula.
* Se sugiere revisar el cheque electrónico. Es importante investigar por qué este método genera tantas cancelaciones y sugerir activamente a los clientes que cambien al pago automático.
* Se debe fidelizar al cliente digital. Ya que la facturación electrónica facilita la cancelación, necesitamos buscar nuevas formas de recordarles el valor del servicio para que decidan quedarse.



# Limpieza y Transformación de Datos

In [ ]:
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1}).astype(object)
df["Churn"] = pd.to_numeric(df["Churn"], errors="coerce")

* Manipulamos el formato de los valores de la variable objetivo para que el modelo interprete de mejor manera. YES/NO -> 1/0

### Pipelines


In [ ]:
pipeline_numeric = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5, weights="uniform")),
    ('scaler', StandardScaler())
])
pipeline_categorical = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
pipeline_preprocessor = ColumnTransformer([
    ("numeric", pipeline_numeric, df.select_dtypes(include=['int64', 'float64']).columns.drop(["Churn"])),
    ("categorical", pipeline_categorical, df.select_dtypes(include=['object']).columns.drop(['customerID']))
])

* Para las variables numericas utilizamos KNN ya que hay pocos registros con nulos y la existencia de correlacion entre las variables que pueden justificar su valor. StandardScaler para estandarizar la variables porque tienen magnitudes diferentes.

* Para las variables categoricas utilizamos SimpleImputer (a pesar de no haber nulos en categoricas) a pesar del riesgo de inflar o desproporcionar las distribuciones. OneHotEncoder ya que la variables cat. son nominales por tanto no tienen un orden jerarquico.

* Descartamos del preprocesador las variables Churn y customerID, porque no aportan al modelo y que churn es el objetivo y customerID es un identificador personal por registro.

In [ ]:
X_crudo = df.drop(['Churn', 'customerID'], axis=1) 
matriz_limpia = pipeline_preprocessor.fit_transform(X_crudo)

if hasattr(matriz_limpia, "toarray"):
    matriz_limpia = matriz_limpia.toarray()

df_limpia = pd.DataFrame(
    matriz_limpia,
    columns=pipeline_preprocessor.get_feature_names_out()
)
df_limpia['Churn'] = df['Churn'].values
df_limpia.head()

In [ ]:
df_limpia.to_csv('../data/Telco_Customer_Churn_limpio.csv', index = False, encoding = "utf-8")

In [ ]:
df_limpia.info()